# E-commerce Profitability Analysis
##  01 — Data Preparation & Validation

**Objective:** Import the raw datasets, verify their grain and keys, assess data quality, reconcile financial calculations, create only analysis-required features, and validate analysis-ready datasets.



In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv("../data/orders.csv")
products = pd.read_csv("../data/products.csv")
marketing = pd.read_csv("../data/marketing_spend.csv")

print("Orders:", orders.shape)
print("Products:", products.shape)
print("Marketing:", marketing.shape)

Orders: (2000, 20)
Products: (207, 9)
Marketing: (144, 10)


### Data Understanding :
Confirm shape, columns, data types, grain, and representative records before cleaning.

In [2]:
for name, df in {"orders": orders, "products": products, "marketing": marketing}.items():
    print(f"\n{name.upper()} — shape={df.shape}")
    print(df.dtypes)
    display(df.head())


ORDERS — shape=(2000, 20)
order_id             object
customer_id          object
order_date           object
channel              object
payment_method       object
region               object
items_ordered         int64
primary_category     object
gross_revenue       float64
discount_pct          int64
discount_amount     float64
shipping_cost       float64
product_cost        float64
platform_fee        float64
transaction_fee     float64
returned             object
refund_amount       float64
net_revenue         float64
total_costs         float64
profit              float64
dtype: object


,order_id,customer_id,order_date,channel,payment_method,region,items_ordered,primary_category,gross_revenue,discount_pct,discount_amount,shipping_cost,product_cost,platform_fee,transaction_fee,returned,refund_amount,net_revenue,total_costs,profit
0,ORD-000001,C-0617,2025-06-24,Mobile App,Gift Card,Northeast,2,Electronics,261.39,0,0.00,22.28,92.30,0.00,7.88,No,0.00,261.39,122.46,138.93
1,ORD-000002,C-0720,2025-08-23,Website,Credit Card,Midwest,1,Beauty,24.02,0,0.00,3.98,10.75,0.00,1.00,No,0.00,24.02,15.73,8.29
2,ORD-000003,C-0189,2025-07-20,Website,Credit Card,Southeast,2,Toys,52.60,0,0.00,14.60,31.41,0.00,1.83,No,0.00,52.60,47.84,4.76
3,ORD-000004,C-0058,2025-07-17,Mobile App,Debit Card,Southeast,2,Sports,30.63,5,1.53,30.98,16.27,0.00,1.14,No,0.00,29.10,48.39,-19.29
4,ORD-000005,C-0580,2025-10-11,Marketplace,Credit Card,Northeast,1,Sports,11.99,25,3.00,10.30,6.88,1.35,0.56,No,0.00,8.99,19.09,-10.10



PRODUCTS — shape=(207, 9)
product_id                 object
product_name               object
category                   object
sub_category               object
unit_cost                 float64
selling_price             float64
shipping_cost_per_unit    float64
weight_lbs                float64
supplier                   object
dtype: object


,product_id,product_name,category,sub_category,unit_cost,selling_price,shipping_cost_per_unit,weight_lbs,supplier
0,SKU-0001,Headphones - Variant 1,Electronics,Headphones,29.44,56.08,13.42,12.40,Supplier A
1,SKU-0002,Headphones - Variant 2,Electronics,Headphones,26.92,91.57,15.50,15.00,Supplier B
2,SKU-0003,Headphones - Variant 3,Electronics,Headphones,15.91,43.34,9.58,7.60,Supplier C
3,SKU-0004,Headphones - Variant 4,Electronics,Headphones,7.13,24.31,5.74,2.80,Supplier B
4,SKU-0005,Headphones - Variant 5,Electronics,Headphones,43.05,135.82,5.82,2.90,Supplier C



MARKETING — shape=(144, 10)
month                  object
platform               object
spend                 float64
impressions             int64
clicks                  int64
conversions             int64
revenue_attributed    float64
cpc                   float64
cpa                   float64
roas                  float64
dtype: object


,month,platform,spend,impressions,clicks,conversions,revenue_attributed,cpc,cpa,roas
0,2024-01,Google Ads,"2,676.14",303594,13271,500,"43,063.16",0.20,5.35,16.09
1,2024-01,Facebook Ads,"1,971.43",338022,7163,333,"15,950.86",0.28,5.92,8.09
2,2024-01,Instagram Ads,"2,418.95",561361,18729,748,"54,454.46",0.13,3.23,22.51
3,2024-01,TikTok Ads,"1,141.70",554273,6967,346,"22,369.23",0.16,3.30,19.59
4,2024-01,Email Marketing,868.82,29509,1215,58,"4,223.97",0.72,14.98,4.86


### Expected keys
- `orders`: one order per `order_id`
- `products`: one product per `product_id`
- `marketing`: one record per `month` + `platform`

In [3]:
print("Unique order_id:", orders["order_id"].nunique(), "/", len(orders))
print("Unique product_id:", products["product_id"].nunique(), "/", len(products))
print("Unique month-platform:", marketing[["month", "platform"]].drop_duplicates().shape[0], "/", len(marketing))

Unique order_id: 2000 / 2000
Unique product_id: 207 / 207
Unique month-platform: 144 / 144


### Data-Quality Assessment
Check missing values, complete duplicates, key duplicates, categorical consistency, and invalid numeric values. 

In [4]:
# 1. Dataset shape
# ------------------------------------------------------------

print("\n1. DATASET SHAPES")

print(f"Orders:    {orders.shape[0]} rows × {orders.shape[1]} columns")
print(f"Products:  {products.shape[0]} rows × {products.shape[1]} columns")
print(f"Marketing: {marketing.shape[0]} rows × {marketing.shape[1]} columns")


1. DATASET SHAPES
Orders:    2000 rows × 20 columns
Products:  207 rows × 9 columns
Marketing: 144 rows × 10 columns


In [5]:
# 2. Missing values
# ------------------------------------------------------------

print("\n2. MISSING VALUES")

for name, df in {
    "Orders": orders,
    "Products": products,
    "Marketing": marketing
}.items():

    missing = df.isna().sum()
    missing = missing[missing > 0]

    print(f"\n{name}:")

    if missing.empty:
        print("No missing values found.")
    else:
        print(missing)


2. MISSING VALUES

Orders:
No missing values found.

Products:
No missing values found.

Marketing:
No missing values found.


In [6]:
# 3. Complete duplicate rows
# ------------------------------------------------------------

print("\n3. DUPLICATE ROWS")

print("Orders duplicates:",
    orders.duplicated().sum())

print("Products duplicates:",
    products.duplicated().sum())

print("Marketing duplicates:",
    marketing.duplicated().sum())


3. DUPLICATE ROWS
Orders duplicates: 0
Products duplicates: 0
Marketing duplicates: 0


In [7]:
# 4. Business-key uniqueness
# ------------------------------------------------------------

print("\n4. BUSINESS KEY VALIDATION")

print(
    "Duplicate order_id:",
    orders["order_id"].duplicated().sum()
)

print(
    "Duplicate product_id:",
    products["product_id"].duplicated().sum()
)

print(
    "Duplicate month-platform combinations:",
    marketing.duplicated(
        subset=["month", "platform"]
    ).sum()
)


4. BUSINESS KEY VALIDATION
Duplicate order_id: 0
Duplicate product_id: 0
Duplicate month-platform combinations: 0


In [8]:
# 5. Data types
# ------------------------------------------------------------

print("\n5. DATA TYPES")

print("\nOrders:")
print(orders.dtypes)




5. DATA TYPES

Orders:
order_id             object
customer_id          object
order_date           object
channel              object
payment_method       object
region               object
items_ordered         int64
primary_category     object
gross_revenue       float64
discount_pct          int64
discount_amount     float64
shipping_cost       float64
product_cost        float64
platform_fee        float64
transaction_fee     float64
returned             object
refund_amount       float64
net_revenue         float64
total_costs         float64
profit              float64
dtype: object


In [9]:
print("\nProducts:")
print(products.dtypes)




Products:
product_id                 object
product_name               object
category                   object
sub_category               object
unit_cost                 float64
selling_price             float64
shipping_cost_per_unit    float64
weight_lbs                float64
supplier                   object
dtype: object


In [10]:
print("\nMarketing:")
print(marketing.dtypes)


Marketing:
month                  object
platform               object
spend                 float64
impressions             int64
clicks                  int64
conversions             int64
revenue_attributed    float64
cpc                   float64
cpa                   float64
roas                  float64
dtype: object


In [11]:
# 6. Negative-value checks
# ------------------------------------------------------------

print("\n6. INVALID NEGATIVE VALUES")

order_nonnegative_columns = [
    "items_ordered",
    "gross_revenue",
    "discount_pct",
    "discount_amount",
    "shipping_cost",
    "product_cost",
    "platform_fee",
    "transaction_fee",
    "refund_amount",
    "total_costs"
]

for col in order_nonnegative_columns:
    count = (orders[col] < 0).sum()
    print(f"Orders - {col}: {count}")


product_nonnegative_columns = [
    "unit_cost",
    "selling_price",
    "shipping_cost_per_unit",
    "weight_lbs"
]

for col in product_nonnegative_columns:
    count = (products[col] < 0).sum()
    print(f"Products - {col}: {count}")


marketing_nonnegative_columns = [
    "spend",
    "impressions",
    "clicks",
    "conversions",
    "revenue_attributed",
    "cpc",
    "cpa",
    "roas"
]

for col in marketing_nonnegative_columns:
    count = (marketing[col] < 0).sum()
    print(f"Marketing - {col}: {count}")




6. INVALID NEGATIVE VALUES
Orders - items_ordered: 0
Orders - gross_revenue: 0
Orders - discount_pct: 0
Orders - discount_amount: 0
Orders - shipping_cost: 0
Orders - product_cost: 0
Orders - platform_fee: 0
Orders - transaction_fee: 0
Orders - refund_amount: 0
Orders - total_costs: 0
Products - unit_cost: 0
Products - selling_price: 0
Products - shipping_cost_per_unit: 0
Products - weight_lbs: 0
Marketing - spend: 0
Marketing - impressions: 0
Marketing - clicks: 0
Marketing - conversions: 0
Marketing - revenue_attributed: 0
Marketing - cpc: 0
Marketing - cpa: 0
Marketing - roas: 0


In [12]:
# 7. Categorical-value consistency
# ------------------------------------------------------------

print("\n7. CATEGORICAL VALUES")

categorical_columns = [
    "channel",
    "payment_method",
    "region",
    "primary_category",
    "returned"
]

for col in categorical_columns:
    print(
        f"\n{col}:",
        sorted(orders[col].astype(str).unique())
    )

print(
    "\nProduct categories:",
    sorted(products["category"].astype(str).unique())
)

print(
    "\nMarketing platforms:",
    sorted(marketing["platform"].astype(str).unique())
)



7. CATEGORICAL VALUES

channel: ['Marketplace', 'Mobile App', 'Social Commerce', 'Website']

payment_method: ['Buy Now Pay Later', 'Credit Card', 'Debit Card', 'Gift Card', 'PayPal']

region: ['Midwest', 'Northeast', 'Southeast', 'Southwest', 'West Coast']

primary_category: ['Beauty', 'Books', 'Clothing', 'Electronics', 'Food & Beverage', 'Home & Kitchen', 'Sports', 'Toys']

returned: ['No', 'Yes']

Product categories: ['Beauty', 'Books', 'Clothing', 'Electronics', 'Food & Beverage', 'Home & Kitchen', 'Sports', 'Toys']

Marketing platforms: ['Email Marketing', 'Facebook Ads', 'Google Ads', 'Influencer', 'Instagram Ads', 'TikTok Ads']


In [13]:
# 8. Date coverage
# ------------------------------------------------------------

print("\n8. DATE COVERAGE")

print(
    "Order date range:",
    orders["order_date"].min(),
    "to",
    orders["order_date"].max()
)

print(
    "Marketing date range:",
    marketing["month"].min(),
    "to",
    marketing["month"].max()
)



8. DATE COVERAGE
Order date range: 2024-01-28 to 2026-01-11
Marketing date range: 2024-01 to 2025-12


In [14]:
# checks the distinct values in important categorical columns.
for col in [
    "channel",
    "payment_method",
    "region",
    "primary_category",
    "returned"
]:
    print(
        f"{col}:",
        sorted(orders[col].astype(str).unique())
    )

channel: ['Marketplace', 'Mobile App', 'Social Commerce', 'Website']
payment_method: ['Buy Now Pay Later', 'Credit Card', 'Debit Card', 'Gift Card', 'PayPal']
region: ['Midwest', 'Northeast', 'Southeast', 'Southwest', 'West Coast']
primary_category: ['Beauty', 'Books', 'Clothing', 'Electronics', 'Food & Beverage', 'Home & Kitchen', 'Sports', 'Toys']
returned: ['No', 'Yes']


In [15]:
# check product category
print(
    "Product categories:",
    sorted(products["category"].astype(str).unique())
)

Product categories: ['Beauty', 'Books', 'Clothing', 'Electronics', 'Food & Beverage', 'Home & Kitchen', 'Sports', 'Toys']


In [16]:
# check marketing platform
print(
    "Marketing platforms:",
    sorted(marketing["platform"].astype(str).unique())
)

Marketing platforms: ['Email Marketing', 'Facebook Ads', 'Google Ads', 'Influencer', 'Instagram Ads', 'TikTok Ads']


#### Verify order-level total costs

In [17]:
# Calculate expected total cost for each order
orders["calculated_total_costs"] = (
    orders["product_cost"]
    + orders["shipping_cost"]
    + orders["platform_fee"]
    + orders["transaction_fee"]
)

# Check whether calculated cost matches reported total_costs
orders["costs_match"] = np.isclose(
    orders["calculated_total_costs"],
    orders["total_costs"],
    atol=0.01
)

# Validation summary
print("Total orders:", len(orders))
print("Matching orders:", orders["costs_match"].sum())
print("Mismatched orders:", (~orders["costs_match"]).sum())

Total orders: 2000
Matching orders: 2000
Mismatched orders: 0


In [18]:
# 9. Financial consistency
# ------------------------------------------------------------

print("\n9. FINANCIAL CONSISTENCY")


# Total cost check
calculated_total_costs = (
    orders["product_cost"]
    + orders["shipping_cost"]
    + orders["platform_fee"]
    + orders["transaction_fee"]
)

cost_match = np.isclose(
    calculated_total_costs,
    orders["total_costs"],
    atol=0.01
)

print(
    "Total cost mismatches:",
    (~cost_match).sum()
)


# Net revenue check
calculated_net_revenue = (
    orders["gross_revenue"]
    - orders["discount_amount"]
    - orders["refund_amount"]
)

net_revenue_match = np.isclose(
    calculated_net_revenue,
    orders["net_revenue"],
    atol=0.01
)

print(
    "Net revenue mismatches:",
    (~net_revenue_match).sum()
)


# Profit check
calculated_profit = (
    orders["net_revenue"]
    - orders["total_costs"]
)

profit_match = np.isclose(
    calculated_profit,
    orders["profit"],
    atol=0.01
)

print(
    "Profit mismatches:",
    (~profit_match).sum()
)




9. FINANCIAL CONSISTENCY
Total cost mismatches: 0
Net revenue mismatches: 0
Profit mismatches: 0


In [19]:
# 10. Return/refund consistency
# ------------------------------------------------------------

print("\n10. RETURN / REFUND CONSISTENCY")

returned_without_refund = orders[
    (orders["returned"] == True)
    & (orders["refund_amount"] <= 0)
]

nonreturned_with_refund = orders[
    (orders["returned"] == False)
    & (orders["refund_amount"] > 0)
]

print(
    "Returned orders without refund:",
    len(returned_without_refund)
)

print(
    "Non-returned orders with refund:",
    len(nonreturned_with_refund)
)



10. RETURN / REFUND CONSISTENCY
Returned orders without refund: 0
Non-returned orders with refund: 0


In [20]:
# 11. Category consistency across datasets
# ------------------------------------------------------------

print("\n11. CATEGORY CONSISTENCY")

order_categories = set(
    orders["primary_category"].dropna().unique()
)

product_categories = set(
    products["category"].dropna().unique()
)

missing_in_products = (
    order_categories - product_categories
)

missing_in_orders = (
    product_categories - order_categories
)

print(
    "Order categories missing from products:",
    missing_in_products
)

print(
    "Product categories not found in orders:",
    missing_in_orders
)


print("\n" + "=" * 60)
print("DATA QUALITY ASSESSMENT COMPLETE")
print("=" * 60)


11. CATEGORY CONSISTENCY
Order categories missing from products: set()
Product categories not found in orders: set()

DATA QUALITY ASSESSMENT COMPLETE


In [21]:
assert orders["costs_match"].all(), \
    "Validation failed: Some orders have incorrect total costs."

print("✓ Validation passed: All order-level costs reconcile correctly.")

✓ Validation passed: All order-level costs reconcile correctly.


### Data Cleaning

Based on the data-quality assessment, only necessary cleaning operations are
performed.

The assessment found no missing values, complete duplicate records, or duplicate
business keys requiring removal. Therefore, no records are dropped or imputed.

The cleaning stage focuses on:

- Ensuring date fields use appropriate datetime types
- Preserving valid business observations

In [22]:
# correct date type
orders["order_date"] = pd.to_datetime(
    orders["order_date"],
    errors="raise"
)

marketing["month"] = pd.to_datetime(
    marketing["month"],
    errors="raise"
)

print("Date fields converted successfully.")

print("order_date:", orders["order_date"].dtype)
print("month:", marketing["month"].dtype)

Date fields converted successfully.
order_date: datetime64[ns]
month: datetime64[ns]


In [23]:
print("=" * 50)
print("DATA CLEANING COMPLETE")
print("=" * 50)

print(f"Orders:    {orders.shape}")
print(f"Products:  {products.shape}")
print(f"Marketing: {marketing.shape}")

DATA CLEANING COMPLETE
Orders:    (2000, 22)
Products:  (207, 9)
Marketing: (144, 10)


###  Data Transformation 

Additional analysis-ready variables are created from existing fields to support
time-based, profitability, return, and marketing analysis.

Only variables directly relevant to the project requirements are created.
Original financial values are preserved.

In [24]:
orders["order_year"] = (
    orders["order_date"].dt.year
)

orders["order_month"] = (
    orders["order_date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

orders["order_month_name"] = (
    orders["order_date"].dt.month_name()
)

print(
    orders[
        [
            "order_date",
            "order_year",
            "order_month",
            "order_month_name"
        ]
    ].head()
)

  order_date  order_year order_month order_month_name
0 2025-06-24        2025  2025-06-01             June
1 2025-08-23        2025  2025-08-01           August
2 2025-07-20        2025  2025-07-01             July
3 2025-07-17        2025  2025-07-01             July
4 2025-10-11        2025  2025-10-01          October


In [25]:
print(
    "Returned values:",
    orders["returned"].unique()
)

Returned values: ['No' 'Yes']


In [26]:
# Order-level profit margin
orders["profit_margin"] = np.where(
    orders["net_revenue"] != 0,
    orders["profit"] / orders["net_revenue"],
    np.nan
)
print(orders["profit_margin"])

0       0.53
1       0.35
2       0.09
3      -0.66
4      -1.12
        ... 
1995    0.23
1996    0.38
1997     NaN
1998    0.10
1999     NaN
Name: profit_margin, Length: 2000, dtype: float64


In [27]:
orders[
    [
        "order_id",
        "net_revenue",
        "profit",
        "profit_margin"
    ]
].head()

,order_id,net_revenue,profit,profit_margin
0,ORD-000001,261.39,138.93,0.53
1,ORD-000002,24.02,8.29,0.35
2,ORD-000003,52.60,4.76,0.09
3,ORD-000004,29.10,-19.29,-0.66
4,ORD-000005,8.99,-10.10,-1.12


In [28]:
# marketing time features
marketing["year"] = (
    marketing["month"].dt.year
)

marketing["month_number"] = (
    marketing["month"].dt.month
)

marketing["month_name"] = (
    marketing["month"].dt.month_name()
)

In [29]:
marketing[
    [
        "month",
        "year",
        "month_number",
        "month_name"
    ]
].head()

,month,year,month_number,month_name
0,2024-01-01,2024,1,January
1,2024-01-01,2024,1,January
2,2024-01-01,2024,1,January
3,2024-01-01,2024,1,January
4,2024-01-01,2024,1,January


In [30]:
print("=" * 50)
print("Data Transforming COMPLETE")
print("=" * 50)

print("\nNew order features:")
print([
    "order_year",
    "order_month",
    "order_month_name",
    "is_returned",
    "profit_margin"
])

print("\nNew marketing features:")
print([
    "year",
    "month_number",
    "month_name"
])

Data Transforming COMPLETE

New order features:
['order_year', 'order_month', 'order_month_name', 'is_returned', 'profit_margin']

New marketing features:
['year', 'month_number', 'month_name']


### Data Validation

In [31]:
order_required_columns = [
    "order_id",
    "order_date",
    "channel",
    "primary_category",
    "gross_revenue",
    "discount_amount",
    "refund_amount",
    "net_revenue",
    "product_cost",
    "shipping_cost",
    "platform_fee",
    "transaction_fee",
    "total_costs",
    "profit"
]

product_required_columns = [
    "product_id",
    "product_name",
    "category",
    "unit_cost",
    "selling_price"
]

marketing_required_columns = [
    "month",
    "platform",
    "spend",
    "clicks",
    "conversions",
    "revenue_attributed"
]

print(
    "Orders required-field missing values:",
    orders[order_required_columns]
    .isna()
    .sum()
    .sum()
)

print(
    "Products required-field missing values:",
    products[product_required_columns]
    .isna()
    .sum()
    .sum()
)

print(
    "Marketing required-field missing values:",
    marketing[marketing_required_columns]
    .isna()
    .sum()
    .sum()
)

Orders required-field missing values: 0
Products required-field missing values: 0
Marketing required-field missing values: 0


In [32]:
print(
    "Duplicate order rows:",
    orders.duplicated().sum()
)

print(
    "Duplicate product rows:",
    products.duplicated().sum()
)

print(
    "Duplicate marketing rows:",
    marketing.duplicated().sum()
)

print(
    "\nDuplicate order IDs:",
    orders["order_id"].duplicated().sum()
)

print(
    "Duplicate product IDs:",
    products["product_id"].duplicated().sum()
)

print(
    "Duplicate month-platform keys:",
    marketing.duplicated(
        subset=["month", "platform"]
    ).sum()
)

Duplicate order rows: 0
Duplicate product rows: 0
Duplicate marketing rows: 0

Duplicate order IDs: 0
Duplicate product IDs: 0
Duplicate month-platform keys: 0


In [33]:
assert pd.api.types.is_datetime64_any_dtype(
    orders["order_date"]
)

assert pd.api.types.is_datetime64_any_dtype(
    orders["order_month"]
)

assert pd.api.types.is_datetime64_any_dtype(
    marketing["month"]
)

print("✓ Date validation passed.")

✓ Date validation passed.


In [34]:
assert orders[order_required_columns].isna().sum().sum() == 0, \
    "Missing required values found in orders."

assert products[product_required_columns].isna().sum().sum() == 0, \
    "Missing required values found in products."

assert marketing[marketing_required_columns].isna().sum().sum() == 0, \
    "Missing required values found in marketing."


assert orders["order_id"].is_unique, \
    "Duplicate order_id values found."

assert products["product_id"].is_unique, \
    "Duplicate product_id values found."

assert not marketing.duplicated(
    subset=["month", "platform"]
).any(), \
    "Duplicate month-platform keys found."




print("=" * 60)
print("FINAL DATA VALIDATION PASSED")
print("All datasets are ready for analysis.")
print("=" * 60)

FINAL DATA VALIDATION PASSED
All datasets are ready for analysis.


In [36]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   month               144 non-null    datetime64[ns]
 1   platform            144 non-null    object        
 2   spend               144 non-null    float64       
 3   impressions         144 non-null    int64         
 4   clicks              144 non-null    int64         
 5   conversions         144 non-null    int64         
 6   revenue_attributed  144 non-null    float64       
 7   cpc                 144 non-null    float64       
 8   cpa                 144 non-null    float64       
 9   roas                144 non-null    float64       
 10  year                144 non-null    int32         
 11  month_number        144 non-null    int32         
 12  month_name          144 non-null    object        
dtypes: datetime64[ns](1), float64(5), int32(2), int64(

### Export Analysis-Ready Datasets

In [35]:
orders.to_csv(
    "../data/processed/orders.csv",
    index=False
)

products.to_csv(
    "../data/processed/products.csv",
    index=False
)

marketing.to_csv(
    "../data/processed/marketing.csv",
    index=False
)